# 3 · A domain that is not browsing

The claim is that this model is not about browsers. This notebook tests that
claim by building something with no browser anywhere in it: **a document
extraction pipeline** with a real fan-out, a real join, and two independent
extractors that get reconciled.

The point is not the domain. It is that the *only* thing that changes is the
data — manifests, stages, edges — while every mechanism (completeness, typing,
policy, search, evidence, receipts) is unchanged.

In [1]:
# Nothing here needs a browser, a model or a network. The core is stdlib-only.
try:
    import browsergraph  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "https://github.com/aidonerightcorp/browsergraph/releases/"
                    "download/v0.3.0/browsergraph-0.3.0-py3-none-any.whl"],
                   check=True)

import browsergraph as bg
print("browsergraph", bg.__version__)

browsergraph 0.3.0


## The task, decomposed

Ask four questions, in this order:

1. What must be true at the end, and **what would prove it independently**?
2. Working backwards, what must be true before that?
3. For each requirement: what type goes in, what comes out?
4. What could perform it? *If the answer is one thing, it is not a decision yet.*

```
                    ┌─ rules extractor ─┐
load → detect → OCR ┤                   ├→ reconcile → verify
                    └─ model extractor ─┘
```

Two extractors on purpose. Independent producers whose disagreements are
information — a reconciler that sees both can flag a conflict that either alone
would report as confident truth.

In [2]:
from browsergraph import (Edge, NodeManifest, ParameterSpec, PortSpec,
                          StageDefinition, WorkbenchDefinition,
                          expand_node_candidates)

PRIOR = {"source": "illustrative-prior", "evidence": 0}

def node(node_id, description, capability, ins, outs, *, params=(),
         permissions=(), quality=0.9, latency=50, cost=0.0,
         deterministic=True, roles=("transform",), is_a=None):
    return NodeManifest(
        id=node_id, kind=node_id.split(".")[-1], description=description,
        roles=roles, capabilities=(capability,),
        inputs=tuple(PortSpec(n, t, semantic=s) for n, t, s in ins),
        outputs=tuple(PortSpec(n, t, semantic=s) for n, t, s in outs),
        parameters=params, permissions=permissions,
        runtime={"deterministic": deterministic, **({"is_a": is_a} if is_a else {})},
        metrics={**PRIOR, "quality": quality, "latency_ms": latency,
                 "cost_usd": cost},
    ).assert_valid()

def choice(name, *values):
    return ParameterSpec(name, "string", default=values[0], choices=values)

NODES = (
    node("doc.load.file", "Reads an authorized document from disk.", "load",
         [("ref", "DocRef", "")], [("doc", "Document", "")],
         params=(choice("mode", "single", "batch"),),
         permissions=("filesystem:read",), quality=0.99, latency=15),
    node("doc.load.url", "Fetches a document over HTTP with provenance.", "load",
         [("ref", "DocRef", "")], [("doc", "Document", "")],
         permissions=("network",), quality=0.93, latency=300),

    node("doc.detect.magic", "Identifies the format from its magic bytes.",
         "detect", [("doc", "Document", "")], [("typed", "TypedDocument", "")],
         quality=0.98, latency=4),
    node("doc.detect.extension", "Trusts the file extension.", "detect",
         [("doc", "Document", "")], [("typed", "TypedDocument", "")],
         quality=0.75, latency=1),

    node("doc.text.embedded", "Takes the text layer the PDF already carries.",
         "text", [("typed", "TypedDocument", "")], [("text", "PageText", "prose")],
         quality=0.97, latency=40),
    node("doc.text.ocr", "Reads the page from its pixels.", "text",
         [("typed", "TypedDocument", "")], [("text", "PageText", "prose")],
         params=(choice("engine", "tesseract", "rapidocr", "paddle"),),
         quality=0.86, latency=900, deterministic=False),

    node("doc.rules.regex", "Deterministic patterns, reviewed and versioned.",
         "rules", [("text", "PageText", "prose")], [("fields", "Fields", "extracted")],
         params=(choice("strictness", "strict", "lenient"),),
         quality=0.82, latency=20),
    node("doc.rules.grammar", "A declared grammar with error positions.", "rules",
         [("text", "PageText", "prose")], [("fields", "Fields", "extracted")],
         quality=0.88, latency=60),

    node("doc.model.schema", "A language model constrained to a schema.", "model",
         [("text", "PageText", "prose")], [("fields", "Fields", "extracted")],
         params=(choice("model", "GLM", "Qwen", "DeepSeek"),),
         permissions=("llm",), quality=0.91, latency=1800, cost=0.012,
         deterministic=False, roles=("model",)),
    node("doc.model.field_by_field", "One model call per field, slower and surer.",
         "model", [("text", "PageText", "prose")], [("fields", "Fields", "extracted")],
         params=(choice("model", "GLM", "Qwen"),),
         permissions=("llm",), quality=0.94, latency=4200, cost=0.03,
         deterministic=False, roles=("model",)),

    node("doc.reconcile.prefer_rules", "Rules win; the model fills the gaps.",
         "reconcile",
         [("a", "Fields", "extracted"), ("b", "Fields", "extracted")],
         [("merged", "Fields", "extracted")], quality=0.9, latency=10),
    node("doc.reconcile.agreement", "Keeps only fields both producers agree on.",
         "reconcile",
         [("a", "Fields", "extracted"), ("b", "Fields", "extracted")],
         [("merged", "Fields", "extracted")], quality=0.96, latency=15),

    node("doc.verify.schema", "Checks the merged fields against a schema.",
         "verify", [("fields", "Fields", "extracted")],
         [("out", "VerifiedFields", "extracted")],
         quality=0.9, latency=8, roles=("verifier",)),
    node("doc.verify.source", "Finds every value back in the source text.",
         "verify", [("fields", "Fields", "extracted")],
         [("out", "VerifiedFields", "extracted")],
         quality=0.97, latency=120, roles=("verifier",)),
)

CANDIDATES = expand_node_candidates(NODES)
print(len(NODES), "definitions ->", len(CANDIDATES), "atomic candidates")

14 definitions -> 21 atomic candidates


## Wiring it as a DAG

Note `reconcile`: **two input ports**, one per extractor. That is the shape a
sequence of stages cannot express, and it is not exotic — it is the ordinary way
you combine two independent opinions.

In [3]:
def stage(sid, name, ins, outs, capability, success):
    return StageDefinition(
        id=sid, name=name, required_capabilities=(capability,), success=success,
        inputs=tuple(PortSpec(n, t, semantic=s) for n, t, s in ins),
        outputs=tuple(PortSpec(n, t, semantic=s) for n, t, s in outs),
    ).with_discovered_candidates(NODES, CANDIDATES)

STAGES = (
    stage("load", "Load document", [("ref", "DocRef", "")],
          [("doc", "Document", "")], "load", "the document is readable and hashed"),
    stage("detect", "Detect format", [("doc", "Document", "")],
          [("typed", "TypedDocument", "")], "detect", "the format is known"),
    stage("text", "Get text", [("typed", "TypedDocument", "")],
          [("text", "PageText", "prose")], "text", "text exists for every page"),
    stage("rules", "Deterministic extraction", [("text", "PageText", "prose")],
          [("fields", "Fields", "extracted")], "rules", "fields extracted by rule"),
    stage("model", "Model extraction", [("text", "PageText", "prose")],
          [("fields", "Fields", "extracted")], "model", "fields extracted by model"),
    stage("reconcile", "Reconcile",
          [("a", "Fields", "extracted"), ("b", "Fields", "extracted")],
          [("merged", "Fields", "extracted")], "reconcile",
          "disagreements are resolved or flagged"),
    stage("verify", "Verify", [("fields", "Fields", "extracted")],
          [("out", "VerifiedFields", "extracted")], "verify",
          "an independent check accepted the fields"),
)

EDGES = (Edge("load", "detect"), Edge("detect", "text"),
         Edge("text", "rules"), Edge("text", "model"),
         Edge("rules", "reconcile", to_port="a"),
         Edge("model", "reconcile", to_port="b"),
         Edge("reconcile", "verify"))

doc = WorkbenchDefinition(
    title="Document extraction",
    task="Extract a declared schema from an unknown document",
    success="every field is found back in the source by an independent check",
    nodes=NODES, candidates=CANDIDATES, stages=STAGES, edges=EDGES)

print("valid  :", doc.validate() == [])
print("layers :", doc.layers())
print("chain? :", doc.is_chain)
print(doc.summary())

valid  : True
layers : [['load'], ['detect'], ['text'], ['rules', 'model'], ['reconcile'], ['verify']]
chain? : False
7 stages · 14 definitions · 21 atomic candidates · 1,440 complete routes · 66 adjacent transitions


The two extractors sit in the **same layer** — they are independent, and the
layering says so without anyone drawing it. That is also the parallelism budget:
`parallel_width` tells you how much of this could run at once.

## Every mechanism works unchanged

In [4]:
from browsergraph.policy import Policy, review

offline = Policy(permissions=frozenset({"filesystem", "filesystem:read"}),
                 deterministic_only=True, name="offline-deterministic")
report = review(doc, offline)
print(report.text())

policy 'offline-deterministic'
  load: 2 of 3 candidates eligible
      doc.load.url: needs network — not granted
  detect: 2 of 2 candidates eligible
  text: 1 of 4 candidates eligible
      doc.text.ocr.tesseract.dd88a26f: not deterministic
      doc.text.ocr.rapidocr.5a360af1: not deterministic
      doc.text.ocr.paddle.47e05af3: not deterministic
  rules: 3 of 3 candidates eligible
  model: 0 of 5 candidates eligible
      doc.model.schema.glm.7875e16f: needs llm — not granted
      doc.model.schema.qwen.b8f5bef1: needs llm — not granted
      doc.model.schema.deepseek.f2c1679d: needs llm — not granted
      ... and 2 more blocked
  reconcile: 2 of 2 candidates eligible
  verify: 2 of 2 candidates eligible
  0 complete routes remain — but model has no eligible candidate, so nothing can run


In [5]:
from browsergraph import search
from browsergraph.workbench import OptimizationObjective, OptimizationProfile

accuracy = OptimizationProfile(id="p.accuracy", name="Accuracy first", objectives=(
    OptimizationObjective("quality", "maximize", 0.8),
    OptimizationObjective("latency_ms", "minimize", 0.1),
    OptimizationObjective("cost_usd", "minimize", 0.1)))
cheap = OptimizationProfile(id="p.cheap", name="Cheap", objectives=(
    OptimizationObjective("cost_usd", "minimize", 0.6),
    OptimizationObjective("latency_ms", "minimize", 0.3),
    OptimizationObjective("quality", "maximize", 0.1)))

for profile in (accuracy, cheap):
    p = search.propose(doc, profile, policy=Policy.permissive(), strategy="exhaustive")
    print(f"--- {profile.name} ---")
    print(p.text(doc).split("route metrics")[0].rstrip())
    print()

--- Accuracy first ---
exhaustive search under 'p.accuracy' — score 0.945, better than 99.4% of the reference sample
  Load document                file · single
  Detect format                magic
  Get text                     embedded
  Deterministic extraction     grammar
  Model extraction             schema · GLM
  Reconcile                    agreement
  Verify                       source

--- Cheap ---
exhaustive search under 'p.cheap' — score 0.981, better than 99.4% of the reference sample
  Load document                file · single
  Detect format                magic
  Get text                     embedded
  Deterministic extraction     grammar
  Model extraction             schema · GLM
  Reconcile                    agreement
  Verify                       source



Two objectives, two genuinely different pipelines out of the same registry —
and each says which sub-step it examined, how many were blocked, and what it
cost. Nothing about the search knew this was documents.

## Compile it, and see what it would need

In [6]:
from browsergraph import compile_route

best = search.propose(doc, accuracy, policy=Policy.permissive(),
                      strategy="exhaustive")
plan = compile_route(doc, best.route, source="accuracy-first")
print(plan.text())
print()
print("could run", plan.parallel_width, "steps at once")

plan:a6a67e31a4fadb486ddcfa8348cef883
  7 steps in 6 layers, up to 2 at once
    1. load           file · single   [mode=single]
    2. detect         magic
    3. text           embedded
    4. rules          grammar
    4. model          schema · GLM   [model=GLM]
    5. reconcile      agreement
    6. verify         source
  needs: filesystem:read, llm
  not deterministic — a re-run may differ legitimately

could run 2 steps at once


## Save it and look at it

The workbench is portable data. The studio is one self-contained offline file
that reads it — five synchronized views over the same description.

In [7]:
import pathlib
out = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").is_dir() \
    else pathlib.Path(".")
doc.write_json(str(out / "document-extraction.json"))
doc.write_html(str(out / "document-extraction-studio.html"))
for p in sorted(out.glob("document-extraction*")):
    print(f"{p.name:<38} {p.stat().st_size/1000:>7.0f} KB")

document-extraction-studio.html             53 KB
document-extraction.json                    21 KB


## What this proves, and what it does not

**Does:** the model is not about browsers. A document pipeline with fan-out and
a join uses exactly the same primitives, the same validator, the same policy
gate, the same search, and the same compiler — and the only thing written by
hand was the data.

**Does not:** none of these numbers are measurements. Every metric here carries
`"source": "illustrative-prior"`, which is enforced by a test. They exist to
give the search something to sort by. **Real optimization consumes real
receipts** — see notebook 02 for how those become evidence.

To apply this to your own problem: list the requirements in order, list what
could satisfy each, declare the types, wire the edges, and run
`browsergraph check`. If a stage has only one candidate, it is not a decision
yet — either find the alternatives or fold it into its neighbour.